 TASK 4: Health Query Chatbot with FREE Hugging Face API

# TASK 4 - STEP 1: Import Libraries

In [1]:
# TASK 4 - STEP 1: Import Libraries

print("="*60)
print("TASK 4: HEALTH QUERY CHATBOT (Free API)")
print("="*60)

import requests
import json
import time
import warnings
warnings.filterwarnings('ignore')

print("\n✅ Libraries imported!")
print("✅ Using Hugging Face Free Inference API")

TASK 4: HEALTH QUERY CHATBOT (Free API)

✅ Libraries imported!
✅ Using Hugging Face Free Inference API


# TASK 4 - STEP 2: Define API Function for Free LLM

In [2]:
# TASK 4 - STEP 2: Define API Function for Free LLM

print("\n" + "="*50)
print("SETTING UP FREE API")
print("="*50)

# Free models on Hugging Face (no API key required)
# Using Microsoft DialoGPT - best for conversations
API_URL = "https://api-inference.huggingface.co/models/microsoft/DialoGPT-medium"

def get_llm_response(user_query):
    """Get response from free Hugging Face API"""
    
    # Prompt engineering - making AI act as medical assistant
    prompt = f"""You are a helpful, friendly medical assistant named HealthBot.
    
RULES:
1. Give ONLY general health information
2. NEVER give specific medical advice or diagnosis
3. Always suggest consulting a doctor for serious concerns
4. Be caring and empathetic
5. If unsure, say "Please consult a doctor for medical advice"

User Question: {user_query}

HealthBot Response:"""
    
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_length": 250,
            "temperature": 0.7,
            "do_sample": True,
            "top_p": 0.9,
            "repetition_penalty": 1.2
        }
    }
    
    try:
        print("⏳ Contacting AI model...")
        response = requests.post(API_URL, json=payload, timeout=60)
        
        if response.status_code == 200:
            result = response.json()
            if isinstance(result, list) and len(result) > 0:
                generated = result[0].get('generated_text', '')
                # Extract only the response part
                if "HealthBot Response:" in generated:
                    response_text = generated.split("HealthBot Response:")[-1].strip()
                else:
                    response_text = generated.replace(prompt, "").strip()
                return response_text
        elif response.status_code == 503:
            # Model is loading, wait and retry
            time.sleep(5)
            return get_llm_response(user_query)
        else:
            return None
    except Exception as e:
        print(f"API Error: {e}")
        return None

print("✅ API function ready!")
print(f"📡 Using model: {API_URL}")


SETTING UP FREE API
✅ API function ready!
📡 Using model: https://api-inference.huggingface.co/models/microsoft/DialoGPT-medium


# TASK 4 - STEP 3: Safety Filters

In [3]:
# TASK 4 - STEP 3: Safety Filters

print("\n" + "="*50)
print("SAFETY FILTERS")
print("="*50)

# Emergency keywords - redirect to professional help
EMERGENCY_KEYWORDS = [
    'emergency', 'suicide', 'kill myself', 'want to die', 
    'heart attack', 'stroke', 'severe bleeding', 'unconscious',
    'not breathing', 'chest pain severe', 'poison', 'overdose'
]

# Medical advice keywords - block these
MEDICAL_ADVICE_KEYWORDS = [
    'prescribe', 'diagnose', 'treatment plan', 'surgery',
    'medication dosage', 'injection', 'vaccination schedule'
]

def is_emergency(query):
    """Check if query is an emergency"""
    query_lower = query.lower()
    for keyword in EMERGENCY_KEYWORDS:
        if keyword in query_lower:
            return True
    return False

def is_medical_advice(query):
    """Check if asking for medical advice"""
    query_lower = query.lower()
    for keyword in MEDICAL_ADVICE_KEYWORDS:
        if keyword in query_lower:
            return True
    return False

def health_response(user_query):
    """Generate safe health response"""
    
    # Emergency check
    if is_emergency(user_query):
        return """⚠️ **EMERGENCY DETECTED** ⚠️

This appears to be a medical emergency.

🚨 **PLEASE CALL EMERGENCY SERVICES IMMEDIATELY:**
   • India: 102 or 108
   • USA: 911
   • UK: 999

Go to the nearest hospital right now. Don't wait for online advice.

💙 Take care. Your health is important!"""
    
    # Check for medical advice requests
    if is_medical_advice(user_query):
        return """⚠️ I cannot provide specific medical advice or prescriptions.

✅ I can help with:
   • General health information
   • Common symptoms explanation
   • When to see a doctor
   • Healthy lifestyle tips

Please consult a qualified doctor for medical advice, diagnosis, or treatment."""
    
    # Get response from API
    response = get_llm_response(user_query)
    
    if response:
        # Add safety disclaimer
        return response + "\n\n---\n💙 *Remember: This is general information. Always consult a doctor for medical advice.*"
    else:
        # Fallback responses
        return get_fallback_response(user_query)

def get_fallback_response(query):
    """Fallback responses when API is unavailable"""
    
    responses = {
        "sore throat": "A sore throat is often caused by viral infections, allergies, or dry air. 💡 Try: Warm water with honey, rest, throat lozenges. See a doctor if it persists beyond 3 days.",
        
        "paracetamol": "Paracetamol (acetaminophen) is a common fever/pain reducer. 💊 For children: Always use children's formula. Dosage is based on weight (10-15 mg/kg). ⚠️ Consult a pediatrician before giving to infants under 3 months.",
        
        "fever": "Fever helps fight infection. 🌡️ For mild fever (under 102°F/39°C): Rest, drink fluids, use lukewarm compress. 🚨 Seek medical care if: Fever over 104°F, lasts 3+ days, or child under 3 months.",
        
        "headache": "Common causes: stress, dehydration, lack of sleep, eye strain. 💆 Remedies: Drink water, rest in dark quiet room, apply cold or warm compress. ⚠️ See doctor for severe or persistent headaches.",
        
        "cough": "Cough helps clear airways. 🍯 Natural remedies: Honey (for adults/children over 1), warm tea with ginger, steam inhalation. 🚨 Seek help if: Difficulty breathing, coughing blood, persists over 3 weeks.",
        
        "cold": "Common cold is viral. 🤧 Rest, drink warm fluids, use saline nasal spray, honey for cough. Usually resolves in 7-10 days. See doctor if fever or symptoms worsen.",
        
        "stomach pain": "Possible causes: indigestion, gas, constipation, food poisoning. 🍵 Try: Ginger tea, peppermint, rest, light meals. ⚠️ Seek medical care if: Severe pain, vomiting, blood in stool."
    }
    
    query_lower = query.lower()
    for key, value in responses.items():
        if key in query_lower:
            return value
    
    return "I'm HealthBot! 🤖 I can provide general health information. 💙\n\nPlease ask about:\n• Common symptoms (fever, cough, headache)\n• Home remedies (sore throat, cold)\n• When to see a doctor\n\n⚠️ For emergencies, call emergency services immediately!"

print("✅ Safety filters configured!")
print(f"🚨 Emergency keywords: {len(EMERGENCY_KEYWORDS)}")
print(f"⚠️ Medical advice blockers: {len(MEDICAL_ADVICE_KEYWORDS)}")


SAFETY FILTERS
✅ Safety filters configured!
🚨 Emergency keywords: 12
⚠️ Medical advice blockers: 7


# TASK 4 - STEP 4: Test the Chatbot

In [4]:
# TASK 4 - STEP 4: Test the Chatbot

print("\n" + "="*50)
print("TESTING THE CHATBOT")
print("="*50)

test_queries = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "I have severe chest pain",
    "What is fever?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*40}")
    print(f"Test {i}: {query}")
    print('-'*40)
    response = health_response(query)
    print(f"🤖 HealthBot: {response}")
    print('='*40)
    time.sleep(1)

print("\n✅ Testing complete!")


TESTING THE CHATBOT

Test 1: What causes a sore throat?
----------------------------------------
⏳ Contacting AI model...
🤖 HealthBot: A sore throat is often caused by viral infections, allergies, or dry air. 💡 Try: Warm water with honey, rest, throat lozenges. See a doctor if it persists beyond 3 days.

Test 2: Is paracetamol safe for children?
----------------------------------------
⏳ Contacting AI model...
🤖 HealthBot: Paracetamol (acetaminophen) is a common fever/pain reducer. 💊 For children: Always use children's formula. Dosage is based on weight (10-15 mg/kg). ⚠️ Consult a pediatrician before giving to infants under 3 months.

Test 3: I have severe chest pain
----------------------------------------
⏳ Contacting AI model...
🤖 HealthBot: I'm HealthBot! 🤖 I can provide general health information. 💙

Please ask about:
• Common symptoms (fever, cough, headache)
• Home remedies (sore throat, cold)
• When to see a doctor

⚠️ For emergencies, call emergency services immediately!

Tes

# TASK 4 - STEP 5: Interactive Mode

In [5]:
# TASK 4 - STEP 5: Interactive Mode

print("\n" + "="*60)
print("💙 HEALTHBOT - INTERACTIVE MODE 💙")
print("="*60)
print("\n📋 I can answer general health questions")
print("⚠️ For emergencies, call emergency services")
print("\nCommands:")
print("   • Type 'quit' to exit")
print("   • Type 'help' for topics")
print("   • Type 'emergency' for emergency info")
print("="*60)

conversation_history = []

while True:
    user_input = input("\n❓ You: ").strip()
    
    if user_input.lower() in ['quit', 'exit', 'q', 'bye']:
        print("\n🤖 HealthBot: Take care of yourself! 💙")
        print("Remember: For any medical concerns, consult a doctor.")
        print("Stay healthy! 🌟")
        break
    
    if user_input.lower() == 'help':
        print("\n📚 TOPICS I CAN HELP WITH:")
        print("   • Symptoms: fever, cough, headache, sore throat, cold")
        print("   • Medicines: paracetamol information")
        print("   • Home remedies for common conditions")
        print("   • When to see a doctor")
        print("   • General health questions")
        print("\n💡 Example questions:")
        print("   - 'What causes a sore throat?'")
        print("   - 'How to reduce fever?'")
        print("   - 'When to see a doctor for headache?'")
        continue
    
    if user_input.lower() == 'emergency':
        print("\n🚨 EMERGENCY CONTACTS:")
        print("   📞 INDIA: 102 (Ambulance), 108 (Emergency)")
        print("   📞 USA: 911")
        print("   📞 UK: 999")
        print("\n   🏥 Go to nearest hospital immediately!")
        continue
    
    # Get response
    print("🤖 HealthBot: Thinking...")
    response = health_response(user_input)
    print(f"🤖 HealthBot: {response}")

print("\n" + "="*60)
print("TASK 4 COMPLETED SUCCESSFULLY!")
print("="*60)
print("\n✅ General Health Query Chatbot created!")
print("✅ Free Hugging Face API integrated!")
print("✅ Prompt engineering implemented!")
print("✅ Safety filters and emergency detection added!")
print("✅ Medical advice blocker activated!")
print("\n📁 Files created: task4_health_chatbot.ipynb")


💙 HEALTHBOT - INTERACTIVE MODE 💙

📋 I can answer general health questions
⚠️ For emergencies, call emergency services

Commands:
   • Type 'quit' to exit
   • Type 'help' for topics
   • Type 'emergency' for emergency info
🤖 HealthBot: Thinking...
⏳ Contacting AI model...
🤖 HealthBot: Fever helps fight infection. 🌡️ For mild fever (under 102°F/39°C): Rest, drink fluids, use lukewarm compress. 🚨 Seek medical care if: Fever over 104°F, lasts 3+ days, or child under 3 months.
🤖 HealthBot: Thinking...
⏳ Contacting AI model...
🤖 HealthBot: I'm HealthBot! 🤖 I can provide general health information. 💙

Please ask about:
• Common symptoms (fever, cough, headache)
• Home remedies (sore throat, cold)
• When to see a doctor

⚠️ For emergencies, call emergency services immediately!
🤖 HealthBot: Thinking...
⏳ Contacting AI model...
🤖 HealthBot: I'm HealthBot! 🤖 I can provide general health information. 💙

Please ask about:
• Common symptoms (fever, cough, headache)
• Home remedies (sore throat, co